In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import pandas as pd
import os
import glob
from sklearn.model_selection import train_test_split

# 1. Define Paths
# /kaggle/input/q3-stage3-2026/dataset --> full path to the images and masks
root_dir = os.path.join(path, "dataset")
img_dir = os.path.join(root_dir, "images")                # u need image dir, and masks dir this time!!
mask_dir = os.path.join(root_dir, "masks")

# 2. Get All Filenames (Sorted to ensure alignment)
image_files = sorted(os.listdir(img_dir))
mask_files  = sorted(os.listdir(mask_dir))

# 3. Create DataFrame
# Construct full paths or relative paths
data = []
for img_name in image_files:
        data.append({
            "image_path": os.path.join(img_dir, img_name),
            "mask_path": os.path.join(mask_dir, img_name)
        })

df = pd.DataFrame(data)

# 4. Split Data (80% Train, 20% Test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Total: {len(df)}, Train: {len(train_df)}, Test: {len(test_df)}")


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import numpy as np



class SegDataset(Dataset):
    def __init__(self, df, transform=None, mask_transform=None):
        self.df = df
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get Paths from DataFrame
        row = self.df.iloc[idx]
        img_path = row['image_path']
        mask_path = row['mask_path']

        # 2. Load Image & Mask
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") # Grayscale for masks

        # 3. Apply Transforms
        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        # 3. Remap Mask (we need this because CrossEntropy requires the labels to be consecutive)
        mask = remap_mask(mask)

        return image, mask


from torchvision import transforms
from torch.utils.data import DataLoader

# 1. Define Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resizing to 224 * 224 (what the model 'efficientnet-b1' expects as an input)
    transforms.ToTensor()
])
transform_mask = transforms.Compose([
    transforms.Resize((224, 224)),  # Resizing to 224 * 224 (what the model 'efficientnet-b1' expects as an input)
    transforms.PILToTensor()
])

# 2. Create Datasets
train_dataset = SegDataset(train_df, transform=transform, mask_transform=transform_mask)
test_dataset  = SegDataset(test_df,  transform=transform, mask_transform=transform_mask)

# 3. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

# 4. Check Batch
imgs, masks = next(iter(test_loader))
print(f"Image: {imgs.shape}, Mask: {masks.shape}")

In [ ]:
# TO DO



In [ ]:
# TO DO

In [ ]:
# TO DO

In [ ]:
# TO DO